In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import igraph
import leidenalg

In [ ]:

import numpy as np
import pandas as pd
import anndata
import scanpy as sc
import scipy
import os
import scipy.io as sio
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl


from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator



In [ ]:
# Path to gene expression
data_path = '/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/all-sample/DGE_filtered/'

# Guide path
guide_data_path = '/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/CRISPR_combined/grnaxcell_count.tsv'
all_guides_path='/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/CRISPR/sub1/all-sample/guide_RNAs_unfiltered/all_guides.csv'
#grna_assignment_path = '/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250509_eTN9_TF1-8_Novogene/splitpipe_out/CRISPR_combined/grna_assignments.csv'

# Directory for guide input into SCEPTRE
#sceptre_input_path = '/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250509_eTN9_TF1-8_Novogene/splitpipe_out/CRISPR_combined/sceptre_input'

# Figure directory
sc.settings.figdir = '/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/figures/'

In [ ]:
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=300, figsize=(5,4), format='png')


### Reading in data

In [ ]:
# The DGE_filtered folder contains the expression matrix, genes, and files
# NOTE: split-pipe versions older than 1.1.0 used 'DGE.mtx'
adata = sc.read_mtx(data_path + 'count_matrix.mtx')

# reading in gene and cell data
gene_data = pd.read_csv(data_path + 'all_genes.csv')
cell_meta = pd.read_csv(data_path + 'cell_metadata.csv')

# find genes with nan values and filter
gene_data = gene_data[gene_data.gene_name.notnull()]
notNa = gene_data.index
notNa = notNa.to_list()

# remove genes with nan values and assign gene names
adata = adata[:,notNa]
adata.var = gene_data
adata.var.set_index('gene_name', inplace=True)
adata.var.index.name = None
adata.var_names_make_unique()

# add cell meta data to anndata object
adata.obs = cell_meta
adata.obs.set_index('bc_wells', inplace=True)
adata.obs.index.name = None
adata.obs_names_make_unique()

#sc.pp.filter_cells(adata, min_genes=10)
#sc.pp.filter_genes(adata, min_cells=5)

# Returns the dimensions of the expression matrix (cells, genes)
adata.shape

### Cell QC

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)


# Scanpy will prepend the string in the save argument with "violin"
# and save it to our figure directory defined in the first step.
sc.pl.violin(adata, ['n_genes_by_counts'], save='_n_genes', jitter=0.4)
sc.pl.violin(adata, ['total_counts'], save='_total_counts', jitter=0.4)
sc.pl.violin(adata, ['pct_counts_mt'], save='_mito_pct', jitter=0.4)

In [ ]:
# Filter the data
adata = adata[adata.obs.n_genes_by_counts < 5000,:]
adata = adata[adata.obs.total_counts < 5000,:]
adata = adata[adata.obs.pct_counts_mt < 40,:]
adata.shape # Checking number of cells remaining

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_gene_vs_transcript_counts')
print('median transcript count per cell: ' + str(adata.obs['tscp_count'].median(0)))
print('median gene count per cell: ' + str(adata.obs['gene_count'].median(0)))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prep data
order = ['CS1', 'DC1', 'Dual', 'no_guide']
df = adata.obs[['total_counts', 'sample']].copy()
df = df[df['sample'].isin(order)].dropna()
df['sample'] = pd.Categorical(df['sample'], categories=order, ordered=True)

# y labels with n
counts = df.groupby('sample')['total_counts'].size()
ylabs = [f"{g} (n={counts.get(g, 0):,})" for g in order]

plt.figure(figsize=(7, 4.5))
ax = sns.violinplot(
    data=df, y='sample', x='total_counts',
    order=order, inner='box', cut=0, scale='width', bw_adjust=1.0,
    palette=['#647FBC', '#91ADC8', '#AED6CF', '#FAFDD6']
)

# axis labels
ax.set_xlabel('Total UMI counts per cell', fontsize=16)
ax.set_ylabel('', fontsize=16)

# tick labels
ax.set_yticklabels(ylabs, fontsize=13)        # y tick labels (your custom strings)
ax.tick_params(axis='x', which='both', labelsize=13)  # x tick labels

sns.despine(ax=ax)
plt.tight_layout()
plt.show()



### Import guide UMI count data frame

In [ ]:
# 1. collect the set of bc_wells in cells
valid_bcs = set(adata.obs.index)

valid_bcs = {bc + "__s1" for bc in valid_bcs}

In [ ]:
len(valid_bcs)

In [ ]:
valid_bcs

In [ ]:
tsv_path = guide_data_path

# 2. Build the set of "in cell" bc_wells directly from adata.obs_names
valid_bcs = set(adata.obs_names)

valid_bcs = {bc + "__s1" for bc in valid_bcs}

# 3. Iterate in chunks, keeping only rows whose 'cell' matches one of valid_bcs
chunks = []
chunksize = 1_000_000

for chunk in pd.read_csv(
    tsv_path,
    sep="\t",
    usecols=["gene", "cell", "count"],
    chunksize=chunksize
):
    mask = chunk["cell"].isin(valid_bcs)
    if mask.any():
        # Keep only the needed columns, with 'cell' already matching adata.obs_names
        filtered = chunk.loc[mask, ["gene", "cell", "count"]]
        chunks.append(filtered)

# 4. Concatenate all filtered pieces
if chunks:
    df_long = pd.concat(chunks, ignore_index=True)
else:
    df_long = pd.DataFrame(columns=["gene", "cell", "count"])

# 5. Pivot into wide form: index='cell', columns='gene', values='count'
df_wide = df_long.pivot(index="cell", columns="gene", values="count")

# 6. Fill missing entries with zero
df_wide = df_wide.fillna(0)

print("Resulting shape:", df_wide.shape)
print(df_wide.head())



In [ ]:
df_wide = df_wide.reindex(index=valid_bcs).fillna(0)

In [ ]:
df_wide.shape

In [ ]:
grna_matrix = df_wide

In [ ]:
grna_matrix

### Reindex columns of grna matrix to be the same as all_guides.csv

In [ ]:

#Remove 'gRNA_' prefix in the column names of gRNA matrix
grna_matrix.columns = grna_matrix.columns.str.replace(r'^gRNA_', '', regex=True)

all_guides = pd.read_csv(all_guides_path)
guide_list = all_guides['gene_name'].tolist()

grna_matrix = grna_matrix.reindex(
    columns=guide_list,
    fill_value=0
)

In [ ]:
grna_matrix.shape

In [ ]:
grna_matrix

### Plot guide distribution for each sample

In [ ]:
adata.obs['sample']

In [ ]:
# remove the suffix "__s1" from row names (index)
grna_matrix.index = grna_matrix.index.str.replace(r'__s1$', '', regex=True)
grna_matrix

In [ ]:
# add the sample name to grna_matrix
# Make sure indices match first (drop "__s1" on adata if needed)
adata_idx = adata.obs.index.to_series().astype(str).str.replace(r'__s1$', '', regex=True)
sample = adata.obs['sample'].copy()
sample.index = adata_idx  # reassign the cleaned index

# (If grna_matrix still has "__s1", clean it the same way)
grna_matrix.index = grna_matrix.index.astype(str).str.replace(r'__s1$', '', regex=True)

# Add the sample column by aligned index
grna_matrix = grna_matrix.join(sample.rename('sample'), how='left')

# check mismatches
print("Unmatched cells:", grna_matrix['sample'].isna().sum())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) Sum ALL gRNA columns (i.e., every numeric column), per cell
#    (exclude any existing metadata columns like 'sample' or previously-made totals)
num_df = grna_matrix.drop(columns=['sample', 'total_gRNA'], errors='ignore')
num_df = num_df.select_dtypes(include=[np.number])
grna_matrix['total_gRNA'] = num_df.sum(axis=1)




In [ ]:
# 2) Order categories
order = ['CS1', 'DC1', 'Dual', 'no_guide']
grna_matrix['sample'] = pd.Categorical(grna_matrix['sample'], categories=order, ordered=True)

# 3) Prep data (+1 pseudocount for log plotting)
plot_data = [
    (grna_matrix.loc[grna_matrix['sample']==c, 'total_gRNA'].to_numpy(dtype=float) + 1.0)
    for c in order
]

# 4) Compute means (on raw counts, without pseudocount)
means_raw = grna_matrix.groupby('sample', observed=False)['total_gRNA'].mean()
xpos = np.arange(1, len(order)+1)
y_means_for_plot = [ (means_raw.get(c, np.nan) + 1.0) for c in order ]  # +1 only for plotting on log axis




In [ ]:
#5) Plot

fig, ax = plt.subplots(figsize=(7,5))
ax.boxplot(plot_data, labels=order, showfliers=True, widths=0.6)
ax.set_yscale('log')
ax.set_ylabel('Total gRNA UMI counts per cell (log scale)')
ax.set_xlabel('gRNA primer types')
#ax.set_title('Total gRNA UMI counts per cell by sample')
ax.grid(True, which='both', axis='y', ls='--', lw=0.5)

# mean markers (optional)
ax.scatter(xpos, y_means_for_plot, zorder=3)

# put the mean numbers at the top of the plot (above the axes)
for x, c in zip(xpos, order):
    m = means_raw.get(c, np.nan)
    if np.isfinite(m):
        ax.text(
            x, 1.02, f'{m:.1f}',
            transform=ax.get_xaxis_transform(),  # x in data coords, y in axes fraction
            ha='center', va='bottom', fontsize=9, clip_on=False
        )

# add headroom so labels aren’t clipped
plt.gcf().subplots_adjust(top=0.85)

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 1) Total gRNA per cell (sum all numeric columns)
num_df = grna_matrix.drop(columns=['sample', 'total_gRNA'], errors='ignore').select_dtypes(include=[np.number])
grna_matrix['total_gRNA'] = num_df.sum(axis=1)

# 2) Keep/order groups
order = ['CS1', 'DC1', 'Dual', 'no_guide']
dfp = grna_matrix.loc[grna_matrix['sample'].isin(order), ['sample', 'total_gRNA']].copy()
dfp['sample'] = pd.Categorical(dfp['sample'], categories=order, ordered=True)

# 3) Pseudocount for log plotting (stats are done on raw counts)
dfp['total_gRNA_plot'] = dfp['total_gRNA'].astype(float) + 1.0

# 4) Colors (Tableau-10)
tableau10 = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2']
palette = dict(zip(order, tableau10))

sns.set(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

fig, ax = plt.subplots(figsize=(9, 5))

# Box (no outliers; points come from stripplot)
sns.boxplot(
    data=dfp, x='sample', y='total_gRNA_plot',
    order=order, palette=palette, width=0.55,
    showfliers=False, saturation=1.0, linewidth=1.2, ax=ax
)

# Jittered points
sns.stripplot(
    data=dfp, x='sample', y='total_gRNA_plot',
    order=order, hue='sample', palette=palette,
    jitter=0.25, alpha=0.6, size=3, linewidth=0,
    ax=ax, legend=False
)

# Log y-axis (start at 10^0 since we added +1)
ax.set_yscale('log')
ax.set_ylim(1, None)
ax.yaxis.set_major_locator(mticker.LogLocator(base=10))
ax.yaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2,10)*0.1))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())

# ----- Mean annotations -----
means_raw = dfp.groupby('sample')['total_gRNA'].mean()  # means on RAW counts
xticks = ax.get_xticks()

y_tops = []
for xi, g in zip(xticks, order):
    grp_vals_plot = dfp.loc[dfp['sample'] == g, 'total_gRNA_plot']
    # place label above the highest observed (on log scale, use multiplicative padding)
    y_top = float(np.nanmax(grp_vals_plot))
    y_label = y_top * 1.08
    y_tops.append(y_label)

    m = float(means_raw.get(g, np.nan))
    # mean marker at its plotted position (mean + 1)
    ax.scatter([xi], [m + 1.0], marker='D', s=36, color='black', zorder=5)
    # numeric label above the box
    ax.text(xi, y_label, f'{m:,.0f}', ha='center', va='bottom', fontsize=11)

# extend ylim if labels need headroom
lo, hi = ax.get_ylim()
need = max(y_tops) * 1.05
if need > hi:
    ax.set_ylim(lo, need)

# Labels, ticks, grid
ax.set_xlabel('Primer type', fontsize=16)
ax.set_ylabel('Total gRNA UMI counts per cell (log)', fontsize=16)
ax.tick_params(axis='both', labelsize=12)
ax.grid(True, axis='y', which='both', linestyle='--', linewidth=0.6, alpha=0.6)

plt.tight_layout()
plt.show()

 


### Make dot plot to assess mixing between cells

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load your matrix (replace with your file path if needed)
df = grna_matrix[grna_matrix >= 1].copy()

# Sum total reads from iPSC and K562 for each cell
df["iPSC_total"] = df[[c for c in df.columns if c.startswith("iPSC_")]].sum(axis=1)
df["K562_total"] = df[[c for c in df.columns if c.startswith("K562_")]].sum(axis=1)

In [ ]:
df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

x = df["iPSC_total"].to_numpy()
y = df["K562_total"].to_numpy()

# --- make both axes use the same numeric range and the same tick positions ---
lo = 0
hi = max(np.nanmax(x), np.nanmax(y))
pad = 0.05 * hi
lo, hi = lo, hi + pad

# choose a common set of "nice" ticks and use it on BOTH axes
loc = mticker.MaxNLocator(nbins=6, integer=True)
ticks = loc.tick_values(lo, hi)
hi = ticks[-1]  # extend limits to the last tick so grids align

plt.figure(figsize=(6,6))
plt.scatter(x, y, alpha=0.5)
plt.xlabel("Total iPSC guide UMIs")
plt.ylabel("Total K562 guide UMIs")
plt.title("iPSC vs K562 gRNA UMIs per cell")

# same limits and ticks on both axes
ax = plt.gca()
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xticks(ticks)
ax.set_yticks(ticks)

# equal data aspect -> one unit in x equals one unit in y
ax.set_aspect('equal', adjustable='box')

# grid (major + optional minor)
ax.grid(True, which="major", ls="--", lw=0.5)
ax.xaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(True, which="minor", ls=":", lw=0.3)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogLocator, NullFormatter, FixedLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --- data & filter (either axis ≥ 3) ---
x = np.asarray(df["iPSC_total"], float)
y = np.asarray(df["K562_total"], float)
m = np.isfinite(x) & np.isfinite(y) & ((x >= 3) | (y >= 3))
x, y = x[m], y[m]

# +1 so zeros are plottable on log; clamp to ≥ 1 (10^0 lower bound)
xp = np.clip(x + 1.0, 1.0, None)
yp = np.clip(y + 1.0, 1.0, None)

# limits & bins starting at 10^0
lo = 1.0
hi = float(max(xp.max(), yp.max()))
hi_dec = 10 ** np.ceil(np.log10(hi))
bins = np.logspace(0, np.log10(hi_dec), 60)

fig, ax = plt.subplots(figsize=(8, 8))
divider = make_axes_locatable(ax)

# ↑ make marginals farther: increase pad, optionally tweak size
marg_size = "20%"   # height/width of marginals
marg_pad  = 0.22    # distance between main and marginal axes
ax_top = divider.append_axes("top",   size=marg_size, pad=marg_pad, sharex=ax)
ax_rt  = divider.append_axes("right", size=marg_size, pad=marg_pad, sharey=ax)

# main 2D histogram
H = ax.hist2d(xp, yp, bins=[bins, bins], cmap="viridis", norm=LogNorm(), rasterized=True)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lo, hi_dec); ax.set_ylim(lo, hi_dec)
ax.set_aspect('equal', adjustable='box')
ax.margins(0, 0)
ax.grid(True, which="both", ls="--", lw=0.5, alpha=0.4)

# marginals
ax_top.set_xscale("log"); ax_rt.set_yscale("log")
ax_top.hist(xp, bins=bins, histtype="stepfilled", alpha=0.6)
ax_rt.hist(yp, bins=bins, orientation="horizontal", histtype="stepfilled", alpha=0.6)
ax_top.set_ylabel("Cells"); ax_rt.set_xlabel("Cells")
ax_top.tick_params(axis='x', labelbottom=False); ax_rt.tick_params(axis='y', labelleft=False)
ax_top.grid(False); ax_rt.grid(False)

# identical ticks on all three axes
maj = LogLocator(base=10, numticks=10)
minr = LogLocator(base=10, subs=np.arange(2,10)*0.1, numticks=100)
ax.xaxis.set_major_locator(maj); ax.xaxis.set_minor_locator(minr); ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_locator(maj); ax.yaxis.set_minor_locator(minr); ax.yaxis.set_minor_formatter(NullFormatter())
fig.canvas.draw()  # force tick computation

ax_top.xaxis.set_major_locator(FixedLocator(ax.get_xticks()))
ax_top.xaxis.set_minor_locator(FixedLocator(ax.get_xticks(minor=True)))
ax_top.xaxis.set_minor_formatter(NullFormatter())
ax_rt.yaxis.set_major_locator(FixedLocator(ax.get_yticks()))
ax_rt.yaxis.set_minor_locator(FixedLocator(ax.get_yticks(minor=True)))
ax_rt.yaxis.set_minor_formatter(NullFormatter())

# compact colorbar (shorter and a bit to the right)
cax = inset_axes(ax, width="3.5%", height="35%", loc="lower left",
                 bbox_to_anchor=(1.30, 0.65, 1, 1),  # x_offset, y_offset in axes coords
                 bbox_transform=ax.transAxes, borderpad=0)
cb = plt.colorbar(H[3], cax=cax)
cb.set_label("Cells per bin (log color)")

ax.set_xlabel("Total iPSC guide UMIs (log)", fontsize=14)
ax.set_ylabel("Total K562 guide UMIs (log)", fontsize=14)


n = len(xp)  # number of cells shown

# Put a centered title across the whole figure and include n
fig.suptitle(f"Total guide UMIs per cell in iPSCs vs K562 (ncells={n:,})",
             ha="center", va="top", y=0.945, fontsize=16)


mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["xtick.labelsize"] = 12
mpl.rcParams["ytick.labelsize"] = 12



plt.show()


### Perform statistical test to see if there's difference between primer types

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# ===== 1) Prep data =====
# Sum ALL numeric gRNA columns per cell (ignore metadata)
num_df = grna_matrix.drop(columns=['sample', 'total_gRNA'], errors='ignore').select_dtypes(include=[np.number])
grna_matrix['total_gRNA'] = num_df.sum(axis=1)

# Focus on the three groups
groups = ['CS1', 'DC1', 'Dual']
df = grna_matrix.loc[grna_matrix['sample'].isin(groups), ['sample', 'total_gRNA']].copy()
df['sample'] = pd.Categorical(df['sample'], categories=groups, ordered=True)

# Data arrays per group (raw counts for stats)
arrs = [df.loc[df['sample']==g, 'total_gRNA'].to_numpy(dtype=float) for g in groups]

# ===== 2) Pairwise p-values only (Dunn-Holm if available; otherwise MW + Holm; fallback Bonferroni) =====
pairs_idx = list(combinations(range(len(groups)), 2))  # [(0,1), (0,2), (1,2)]

def holm_adjust(pvals):
    # Holm step-down (no external deps)
    m = len(pvals)
    order = np.argsort(pvals)
    p_sorted = np.array(pvals)[order]
    adj = np.empty(m, dtype=float)
    for k, p in enumerate(p_sorted, start=1):
        adj[k-1] = (m - k + 1) * p
    # cumulative max to enforce monotonicity
    adj = np.maximum.accumulate(adj)
    adj = np.minimum(adj, 1.0)
    out = np.empty(m, dtype=float)
    out[order] = adj
    return out

pair_p_adj = {}
used_method = None

# Try Dunn test
try:
    import scikit_posthocs as sp
    dunn = sp.posthoc_dunn(df, val_col='total_gRNA', group_col='sample', p_adjust='holm')
    for i, j in pairs_idx:
        gi, gj = groups[i], groups[j]
        pair_p_adj[(i, j)] = float(dunn.loc[gi, gj])
    used_method = "Dunn (Holm)"
except Exception:
    # Fallback: Mann–Whitney U (two-sided) and Holm adjust
    try:
        from scipy.stats import mannwhitneyu
        p_raw = []
        for i, j in pairs_idx:
            u, p = mannwhitneyu(arrs[i], arrs[j], alternative='two-sided', method='asymptotic')
            p_raw.append(p)
        p_adj = holm_adjust(p_raw)
        for (i, j), p in zip(pairs_idx, p_adj):
            pair_p_adj[(i, j)] = float(p)
        used_method = "Mann–Whitney U (Holm)"
    except Exception:
        # Last resort: Bonferroni
        from math import comb
        from scipy.stats import mannwhitneyu
        m = len(pairs_idx)
        for i, j in pairs_idx:
            _, p = mannwhitneyu(arrs[i], arrs[j], alternative='two-sided')
            pair_p_adj[(i, j)] = min(1.0, p * m)
        used_method = "Mann–Whitney U (Bonferroni)"

def p_to_stars(p):
    return '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 5e-2 else 'ns'

# ===== 3) Plot with non-overlapping significance bars =====
# Use boxplot on log-scale (visual), stats were done on raw counts
plot_data = [a + 1.0 for a in arrs]  # +1 to allow log( ) when zeros present
xpos = np.arange(1, len(groups)+1)

fig, ax = plt.subplots(figsize=(7,5))
ax.boxplot(plot_data, labels=groups, showfliers=True, widths=0.6)
ax.set_yscale('log')
ax.set_ylabel('Total gRNA counts per cell (+1), log scale')
ax.set_xlabel('sample')
ax.set_title('Pairwise comparisons ({} adjusted p-values)'.format(used_method))
ax.grid(True, which='both', axis='y', ls='--', lw=0.5)

# Determine per-group "top" (on plotting scale) to start brackets
group_tops = [float(np.nanmax(v)) if len(v) else 1.0 for v in plot_data]
ymax = np.nanmax(group_tops) if len(group_tops) else 1.0
ax.set_ylim(top=ymax * 1.5)  # initial headroom

# Helper: add a bracket that avoids overlap (works well on log scale by multiplicative lifting)
def add_sig_bracket(ax, x1, x2, y, star, pad=1.03, text_pad=1.02):
    y_top = y * pad
    ax.plot([x1, x1, x2, x2], [y, y_top, y_top, y], lw=1, c='k')
    if star != 'ns':
        ax.text((x1 + x2)/2, y_top * text_pad, star, ha='center', va='bottom', fontsize=11)

# Layout the bars with collision avoidance
# Sort by span first, then by p-value (shorter + more significant first reduces clutter)
pairs_sorted = sorted(pair_p_adj.items(), key=lambda kv: ((kv[0][1]-kv[0][0]), kv[1]))
used_levels = []  # list of tuples: (y_top, xL, xR)

for (i, j), p in pairs_sorted:
    star = p_to_stars(p)
    if star == 'ns':
        continue  # draw only significant pairs
    x1, x2 = i+1, j+1
    # base height = max top of the two groups
    y = max(group_tops[i], group_tops[j]) * 1.08
    # lift multiplicatively until it doesn't collide with any used level that overlaps in x-range
    while any((x2 >= xL and x1 <= xR) and (abs(np.log(y) - np.log(y_used)) < 0.08)
              for (y_used, xL, xR) in used_levels):
        y *= 1.18
    add_sig_bracket(ax, x1, x2, y, star, pad=1.03, text_pad=1.02)
    used_levels.append((y*1.03, x1, x2))
    # expand ylim if needed
    lo, hi = ax.get_ylim()
    if y*1.08 > hi:
        ax.set_ylim(lo, y*1.25)

plt.show()

# Optional: print a small table of adjusted p-values
pair_table = pd.DataFrame(
    index=groups, columns=groups, data=np.nan, dtype=float
)
for (i, j), p in pair_p_adj.items():
    pair_table.iloc[i, j] = p
    pair_table.iloc[j, i] = p
print("Pairwise adjusted p-values ({}):\n{}".format(used_method, pair_table))




### Identify highly-variable genes and regress out transcript counts

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.25)
sc.pl.highly_variable_genes(adata, save='') # scanpy generates the filename automatically

# Save raw expression values before variable gene subset
adata.raw = adata.copy()

adata = adata[:, adata.var.highly_variable]
sc.pp.regress_out(adata, ['total_counts'])
sc.pp.scale(adata, max_value=10)

### Principal component analysis


In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='') # scanpy generates the filename automatically

### UMAP and Leiden Clustering

In [ ]:
#adata = adata[adata.obs.total_counts > 500,:]
adata.shape

In [ ]:
sc.pp.neighbors(adata, n_neighbors=20, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.2)
sc.pl.umap(adata, color=['leiden'], legend_fontsize=8, save='_leiden')

In [ ]:
sc.pl.umap(adata, color=["SOX2", "NANOG", "GATA1", "KLF1"], ncols=2, color_map='Blues')

In [ ]:
new_cluster_names = [
    "K562",
    "hiPSC",

]
adata.rename_categories("leiden", new_cluster_names)
sc.pl.umap(
    adata, color="leiden", legend_loc="on data", title=""
)

### Visualize iPSC vs K562 gRNA count on UMAP

In [ ]:
grna_matrix

In [ ]:
# Load your matrix (replace with your file path if needed)
import numpy as np
import pandas as pd

df = grna_matrix.copy()

# Work only on guide columns (safer than "all numeric")
guide_cols = df.filter(regex=r'^(iPSC_|K562_)').columns

# Ensure numeric (in case some guide cols were read as 'object'/'category')
df[guide_cols] = df[guide_cols].apply(pd.to_numeric, errors='coerce')

# Option A: treat <1 as 0
df[guide_cols] = df[guide_cols].where(df[guide_cols] >= 3, 0)

# (Option B instead: treat <1 as missing)
# df[guide_cols] = df[guide_cols].where(df[guide_cols] >= 1)

# Totals per cell
df["iPSC_total"] = df.filter(like="iPSC_").sum(axis=1)
df["K562_total"] = df.filter(like="K562_").sum(axis=1)


### Make guide distribution based on leiden

In [ ]:
adata.obs['leiden']

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- 0) Bring leiden onto grna_matrix (match by cell IDs; strip "__s1" if present) ---
grna_matrix.index = grna_matrix.index.astype(str).str.replace(r'__s1$', '', regex=True)
leid = adata.obs['leiden'].copy()
leid.index = adata.obs.index.astype(str).str.replace(r'__s1$', '', regex=True)
grna_matrix = grna_matrix.join(leid.rename('leiden'), how='left')

# --- 1) Total gRNA per cell (sum all numeric guide columns) ---
num_df = grna_matrix.drop(columns=['sample', 'total_gRNA', 'leiden'], errors='ignore').select_dtypes(include=[np.number])
grna_matrix['total_gRNA'] = num_df.sum(axis=1)

# --- 2) Prep plotting frame & categorical orders ---
order = ['CS1', 'DC1', 'Dual', 'no_guide']
horder = ['K562', 'hiPSC']  # given two leiden categories

dfp = grna_matrix.loc[grna_matrix['sample'].isin(order) & grna_matrix['leiden'].isin(horder),
                      ['sample', 'leiden', 'total_gRNA']].copy()
dfp['sample'] = pd.Categorical(dfp['sample'], categories=order, ordered=True)
dfp['leiden'] = pd.Categorical(dfp['leiden'], categories=horder, ordered=True)

# +1 pseudocount for log plotting (stats computed on raw counts)
dfp['total_gRNA_plot'] = dfp['total_gRNA'].astype(float) + 1.0

# --- 3) Colors: use two Tableau-10 hues for the two leiden groups ---
hue_palette = {'K562': '#4E79A7', 'hiPSC': '#F28E2B'}  # blue/orange

sns.set(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})
fig, ax = plt.subplots(figsize=(10, 5.2))

# Boxplot grouped by sample (x) and split by leiden (hue)
sns.boxplot(
    data=dfp, x='sample', y='total_gRNA_plot',
    order=order, hue='leiden', hue_order=horder,
    palette=hue_palette, width=0.55, showfliers=False,
    saturation=1.0, linewidth=1.2, ax=ax
)

# Jittered points per group, split by hue
sns.stripplot(
    data=dfp, x='sample', y='total_gRNA_plot',
    order=order, hue='leiden', hue_order=horder,
    palette=hue_palette, dodge=True, jitter=0.25,
    alpha=0.45, size=2.5, linewidth=0, ax=ax, legend=False
)

# --- 4) Log y-axis & ticks ---
ax.set_yscale('log')
ax.set_ylim(1, None)
ax.yaxis.set_major_locator(mticker.LogLocator(base=10))
ax.yaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2,10)*0.1))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())

# --- 5) Overlay mean diamonds and numeric mean labels per (sample, leiden) ---
means_raw = dfp.groupby(['sample','leiden'])['total_gRNA'].mean()

# Grab the box patches to find x positions; they appear in sample-major, hue-minor order
patches = ax.artists  # one PathPatch per drawn box
n_h = len(horder)

y_need = []
for i, patch in enumerate(patches):
    s_idx = i // n_h
    h_idx = i % n_h
    samp = order[s_idx]
    hue  = horder[h_idx]

    # Skip if this group is missing
    if (samp, hue) not in means_raw.index:
        continue

    # X coord at center of this box
    x_c = patch.get_x() + patch.get_width()/2.0

    # Mean (raw) -> plot position needs +1
    m = float(means_raw.loc[(samp, hue)])
    ax.scatter([x_c], [m + 1.0], marker='D', s=38, color='black', zorder=5)

    # Place numeric label a bit above the highest point in this subgroup (on log scale)
    grp_vals = dfp.loc[(dfp['sample']==samp) & (dfp['leiden']==hue), 'total_gRNA_plot']
    if len(grp_vals):
        y_top = float(np.nanmax(grp_vals))
        y_lab = y_top * 1.08
        ax.text(x_c, y_lab, f'{m:,.0f}', ha='center', va='bottom', fontsize=10)
        y_need.append(y_lab)

# add headroom if labels exceed current ylim
yl = ax.get_ylim()
if y_need and max(y_need)*1.05 > yl[1]:
    ax.set_ylim(yl[0], max(y_need)*1.08)

# --- 6) Labels, legend, grid ---
ax.set_xlabel('Primer type', fontsize=16)
ax.set_ylabel('Total gRNA UMI counts per cell (log)', fontsize=16)
ax.tick_params(axis='both', labelsize=12)
ax.grid(True, axis='y', which='both', linestyle='--', linewidth=0.6, alpha=0.6)

# Clean legend (from the first boxplot call)
leg = ax.legend(title='Cell type', frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()


### Make saturation curves for all 4 samples

In [ ]:
# Requirements: pandas, numpy, scipy, matplotlib
# If needed: pip install pandas numpy scipy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from pathlib import Path

# ---- Put your file paths here (these are your OAK paths as given) ----
files = {
    "CS1": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/CS1/report/tscp_counts_subsampled.csv",
    "DC1": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/DC1/report/tscp_counts_subsampled.csv",
    "Dual": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/Dual/report/tscp_counts_subsampled.csv",
    "no_guide": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/no_guide/report/tscp_counts_subsampled.csv",
}

def load_counts_csv(fp: str) -> pd.DataFrame:
    """
    Reads the CSV with a blank first header, then 'hg38' as second column.
    Ignores the first *row* as requested.
    Returns a DataFrame with columns: ['Read count','Transcript count'].
    """
    # Read with no header, then assign names
    df_raw = pd.read_csv(fp, header=None)
    # Drop the first row (index 0) as requested
    df_raw = df_raw.iloc[1:].copy()
    # The file form shows two columns: (index-like), hg38
    # Ensure numeric and rename
    df_raw.columns = ["Read count", "Transcript count"]
    df_raw["Read count"] = pd.to_numeric(df_raw["Read count"], errors="coerce")
    df_raw["Transcript count"] = pd.to_numeric(df_raw["Transcript count"], errors="coerce")
    df_raw = df_raw.dropna().sort_values("Read count")
    return df_raw

def fit_spline(x: np.ndarray, y: np.ndarray):
    """
    Fit a smoothing spline through all points.
    We use a gentle smoothing factor to avoid overfitting but keep trend.
    """
    # Ensure strictly increasing x for spline; if duplicates, average y
    x_unique, idx = np.unique(x, return_index=True)
    y_unique = y[idx]

    # Smoothing parameter heuristic: small fraction of variance
    s = len(x_unique) * (np.std(y_unique) ** 2) * 0.001
    spline = UnivariateSpline(x_unique, y_unique, s=s)
    return spline

plt.figure(figsize=(7,5), dpi=140)

any_data = False
for label, path in files.items():
    if not Path(path).exists():
        print(f"[WARN] Missing file: {path}")
        continue
    df = load_counts_csv(path)
    if df.empty:
        print(f"[WARN] No data after parsing: {path}")
        continue

    x = df["Read count"].to_numpy()
    y = df["Transcript count"].to_numpy()

    # Fit spline and evaluate on a dense grid
    spline = fit_spline(x, y)
    x_grid = np.linspace(x.min(), x.max(), 400)
    y_fit = spline(x_grid)

    # Plot (points optional; comment out if you want only curves)
    # plt.scatter(x, y, s=12, alpha=0.4)
    plt.plot(x_grid, y_fit, label=label)
    any_data = True

plt.xlabel("Read count")
plt.ylabel("Transcript count")
plt.title("Saturation curves of transcript vs read counts for 4 primer types")
if any_data:
    plt.legend(frameon=False)
else:
    print("[ERROR] No files were found/read. Check paths or upload the CSVs.")
plt.tight_layout()
plt.show()


In [ ]:
# Same as before, but add DISTINCT MARKERS per curve for readability.
# Requirements: pandas, numpy, scipy, matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from pathlib import Path

files = {
    "CS1": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/CS1/report/tscp_counts_subsampled.csv",
    "DC1": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/DC1/report/tscp_counts_subsampled.csv",
    "Dual": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/Dual/report/tscp_counts_subsampled.csv",
    "no_guide": "/oak/stanford/groups/engreitz/Users/tri/Perturb-Seq/analysis/250730_eTN10_Parsev3_mixediPSC_K562/splitpipe_out/WT/sub1/no_guide/report/tscp_counts_subsampled.csv",
}

# Choose distinct markers for each curve
marker_map = {
    "CS1": "o",        # circle
    "DC1": "s",        # square
    "Dual": "^",       # triangle up
    "no_guide": "D",   # diamond
}

def load_counts_csv(fp: str) -> pd.DataFrame:
    df_raw = pd.read_csv(fp, header=None)
    df_raw = df_raw.iloc[1:].copy()  # ignore the first row
    df_raw.columns = ["Read count", "Transcript count"]
    df_raw["Read count"] = pd.to_numeric(df_raw["Read count"], errors="coerce")
    df_raw["Transcript count"] = pd.to_numeric(df_raw["Transcript count"], errors="coerce")
    return df_raw.dropna().sort_values("Read count")

def fit_spline(x: np.ndarray, y: np.ndarray):
    x_unique, idx = np.unique(x, return_index=True)
    y_unique = y[idx]
    s = len(x_unique) * (np.std(y_unique) ** 2) * 0.001
    return UnivariateSpline(x_unique, y_unique, s=s)

plt.figure(figsize=(7,5), dpi=140)

any_data = False
for label, path in files.items():
    if not Path(path).exists():
        print(f"[WARN] Missing file: {path}")
        continue

    df = load_counts_csv(path)
    if df.empty:
        print(f"[WARN] No data after parsing: {path}")
        continue

    x = df["Read count"].to_numpy()
    y = df["Transcript count"].to_numpy()

    spline = fit_spline(x, y)
    x_grid = np.linspace(x.min(), x.max(), 400)
    y_fit = spline(x_grid)

    # Plot fitted curve
    plt.plot(
        x_grid,
        y_fit,
        label=label,
        linestyle="-",
        linewidth=1.8,
        marker=marker_map.get(label, "o"),
        markevery=30,               # place markers every ~30 points along the curve
        markersize=5,
    )

    # Optional: show raw points faintly with the SAME marker (comment out if not needed)
    # plt.plot(x, y, linestyle="none", marker=marker_map.get(label, "o"), markersize=3, alpha=0.4)

    any_data = True

plt.xlabel("Read count")
plt.ylabel("Transcript count")
if any_data:
    # Make legend emphasize markers
    leg = plt.legend(frameon=False, handlelength=2, handletextpad=0.6)
else:
    print("[ERROR] No files were found/read. Check paths or upload the CSVs.")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# --- 1) Make sure indices match (drop the "__s1" suffix if present on either side) ---
adata.obs_names = adata.obs_names.str.replace(r'__s1$', '', regex=True)
df.index = df.index.astype(str).str.replace(r'__s1$', '', regex=True)

# --- 2) Keep just the two columns we want and ensure numeric ---
cols = ["iPSC_total", "K562_total"]
df_tot = df.reindex(columns=cols)
df_tot = df_tot.apply(pd.to_numeric, errors='coerce')

# --- 3) Join into adata.obs by index (cell barcodes) ---
adata.obs = adata.obs.join(df_tot, how='left')

# Optional: sanity check
print("Unmatched iPSC_total:", adata.obs["iPSC_total"].isna().sum())
print("Unmatched K562_total:", adata.obs["K562_total"].isna().sum())



# --- 5) Plot on UMAP (continuous colormap; clip at p99 so outliers don’t dominate) ---
sc.pl.umap(
    adata,
    color=["iPSC_total", "K562_total"],
    color_map="viridis",
    vmin=0, vmax="p99",
    wspace=0.35
)


In [ ]:
adata

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden", standard_scale="var", n_genes=5
)

In [ ]:
#collapse grna_matrix by genes
grna_matrix_by_genes = grna_matrix.groupby(grna_matrix.columns.str.split("_").str[0], axis=1).sum()
grna_matrix_by_genes

In [ ]:
mask_targets_incell = grna_matrix_by_genes > 5
mask_targets_incell

In [ ]:
# count Trues per column
n_cells_per_gene = mask_targets_incell.sum(axis=0)

print(n_cells_per_gene)

### Calculate skew ratio

In [ ]:
from scipy.stats import skew

# assume n_cells_per_gene is your pandas Series of counts

# 1. sort the counts (descending)
sorted_counts = n_cells_per_gene.sort_values(ascending=False)
print("Sorted counts (top 100):")
print(sorted_counts.head(100))

# 2. calculate the skewness of the distribution
distribution_skewness = skew(n_cells_per_gene)
print(f"\nSkewness (SciPy): {distribution_skewness:.3f}")

# you can also use pandas' built-in:
distribution_skewness_pd = n_cells_per_gene.skew()
print(f"Skewness (pandas): {distribution_skewness_pd:.3f}")

# 3. (optional) mean/median ratio as a simple skew ratio
mean_to_median = n_cells_per_gene.mean() / n_cells_per_gene.median()
print(f"\nMean/Median ratio: {mean_to_median:.3f}")